# Starter's Guide
This guide will very simply illustrate how to use the BBT. This will involve:
   - executing an experiment of a metaheuristic optimising a benchmark function, and
   - collecting data during execution. 
   - After the experiment, indicators will be calculated from the experiment data.

In practise one would likely want to use an existing library for experimentation, and many libraries include functionality to log custom statistics during execution. However, using a library in this guide would decrease accessibility, as one would have to be relatively experienced in that library to follow. Thus, we will stick to "plain old Python" for this guide, and try to keep things simple.

## Setup

First, we need a problem to optimise. The Bent Cigar benchmark function will do. We will use our foresight to also define a method to check if a position is within the bounds of the benchmark function. This will be useful later to calculate the `INFEASIBLE%` indicator.

In [1]:
# Bent Cigar function: global best at  [0, 0, 0, ..., 0], fitness = 0
dim = 10
lower_bound = -100
upper_bound = 100
def fitness(x): return x[0] ** 2 + 1_000_000 * sum([x[k] ** 2 for k in range(1, len(x))])
def in_bounds(x): return all([lower_bound <= k <= upper_bound for k in x])


Second, we will throw together a basic gbest inertia weight PSO. We'll define a Particle class to make life easier. Using some foresight again, we will incorporate logging the STN and IN data into the Particle class. Doing it this way is easier than separating this logic. 

In [2]:
def stn(run_substitute, fitness1, position1, fitness2, position2):
    def position_to_tag(fl):
        p = round((fl - lower_bound) / (upper_bound - lower_bound) * 100)
        return "{:03}".format(p)
    position1_string = "".join([position_to_tag(p) for p in position1])
    position2_string = "".join([position_to_tag(p) for p in position2])
    stn_csv.append(f"{run_substitute},{fitness1},{position1_string},{fitness2},{position2_string}")

# yes this is operating on a global variable that does not exist ;) don't worry about it
def interaction_network(index_1, index_2):
    network[index_1][index_2] += 1
    network[index_2][index_1] += 1

In [3]:
from random import random, uniform
w = 0.9; c1= 0.7; c2 = 0.7

class Particle:
    def __init__(self):
        self.position = [uniform(lower_bound,upper_bound) for _ in range(dim)]
        self.velocity = [0] * dim
        self.fitness = fitness(self.position)
        self.personal_best = self.position.copy()
        self.personal_best_fitness = self.fitness
        
    def update_personal_best(self):
        if self.fitness < self.personal_best_fitness:
            self.personal_best = self.position.copy()
            self.personal_best_fitness = self.fitness
    
    def update(self, gb_position, p_i, gbi_i):
        r1 = [random() for _ in range(dim)]
        r2 = [random() for _ in range(dim)]
        old_fitness = self.fitness
        old_position = self.position.copy()
        for k in range(dim):
            cognitive =  c1 * r1[k] * (self.personal_best[k] - self.position[k])
            social = c2 * r2[k] * (gb_position[k] - self.position[k])
            self.velocity[k] = w * self.velocity[k] + social + cognitive
            self.position[k] += self.velocity[k]
        self.fitness = fitness(self.position)
        interaction_network(p_i, gbi_i)
        stn(p_i, old_fitness, old_position, self.fitness, self.position)      

Functions for logging data for the remaining indicators are required. Specifically, we need to log the "mobility", "diversity", "F%" and the best known fitness value. At this point we are also going to define the population size because it will be helpful to calculate the remaining indicators. You might also notice that we are _again_ altering global variables that don't exist yet. This is bad and lazy, but it's easier than the alternatives.

In [4]:
n = 30 # population size

def distance_util(position_1, position_2):
    return sum((position_1[j] - position_2[j]) ** 2 for j in range(dim)) ** 0.5

def mobility(i, gb_position, particle_position):
    mbty.append(f"{i},{distance_util(gb_position, particle_position)}")

def diversity(population):
    x_ = [sum(p.position[j] for p in population) / n for j in range(dim)]
    n_sum = sum([distance_util(population[i].position, x_) for i in range(n)])
    div.append(n_sum / n)

def f_percent(population):
    outside = sum(not in_bounds(p.position) for p in population)
    f_pct.append(outside / n * 100)

def fitness_csv(gb_fitness):
    fitnesses.append(gb_fitness)

## Experiment execution

Put together what we've already done and we can execute the experiment. We will create all those global variables we've been referencing for the data to be collected into. By only defining them now, we can execute this block of code several times without worrying about appending new data to old data. We will also create a population of Particles, which are just a list of Particle objects. The experiment will be run, and data will be collected where necessary. Remember that the STN and IN are updated by each particle's update function.

In [5]:
stn_csv = []
network = [[0] * n] * n
mbty = []
f_pct = []
fitnesses = []
div = []

swarm = [Particle() for _ in range(n)]
global_best = swarm[0].position.copy() # just to start off with
global_best_fitness = swarm[0].fitness
global_best_index = 0
for its in range(500):
    for index in range(n):
        particle = swarm[index]
        particle.update_personal_best()
        if particle.fitness < global_best_fitness:
            global_best_index = index
            mobility(its, global_best, particle.position)
            global_best = particle.position.copy()
            global_best_fitness = particle.fitness
        particle.update(global_best, index, global_best_index)
    fitness_csv(global_best_fitness)
    f_percent(swarm)
    diversity(swarm)

## Logging

The data is ready to be saved to file. Each type of data needs to be handled a little differently, so we'll make sure they're all in the format that the BBT expects. We will be saving these to the current folder (`tutorials`) because we are lazy :)

In [6]:
with open("diversity.csv", "w") as f:
    f.write("iteration,diversity\n")
    for i in range(500):
        f.write(f"{i},{div[i]}\n")
        
with open("fitness.csv", "w") as f:
    f.write("iteration,fitness\n")
    for i in range(500):
        f.write(f"{i},{fitnesses[i]}\n")
        
with open("mobility.csv", "w") as f:
    f.write("iteration,mobility\n")
    for line in mbty:
        f.write(f"{line}\n")
        
with open("f_percent.csv", "w") as f:
    f.write("iteration,f_percent\n")
    for i in range(500):
        f.write(f"{i},{f_pct[i]}\n")
        
with open("stn.csv", "w") as f:
    f.write("Run,Fitness1,Solution1,Fitness2,Solution2\n")
    for line in stn_csv:
        f.write(f"{line}\n")
        
with open("interaction_network.txt", "w") as f:
    empty_network = [[0] * n] * n
    empty_network_string = " ".join(map(str, [digit for sublist in empty_network for digit in sublist]))
    network_string = " ".join(map(str, [digit for sublist in network for digit in sublist]))
    f.write(f"ig:#0 {empty_network_string}\n")
    f.write(f"ig:#1 {network_string}\n")
    
with open("metadata.json", "w") as f:
    f.write('{"total_iterations": "500", "solution_index": "' + str(global_best_index) + '", "global_best_fitness": "0.0"}')

## Indicator calculation

By passing the path of where we saved the data, we can create an Indicators object.

In [16]:
from behavioural_benchmark.indicators import  Indicators

indicators = Indicators("./")

Illustrating that we can now access all the indicator values:

In [17]:
print(indicators.get_DRoC_A())
print(indicators.get_ERT_Diversity())
print(indicators.get_Critical_Diversity())
print(indicators.get_Critical_Fitness())
print(indicators.get_MRoC_B())
print(indicators.get_Critical_Mobility())

-1.274055511741741
56.48680054464482
16.090094095023467
54008467.61851945
-0.060011611325268936
21.482256198386672


In [18]:
print(indicators.get_ntotal_star())
print(indicators.get_nshared_star())

3500
277


In [19]:
print(indicators.get_MID())
print(indicators.get_MGC())
print(indicators.get_SNID())

2.7249190938511325
0.9425026968716289
80.762


In [20]:
print(indicators.get_EXPLORE_percent())
print(indicators.get_INFEASIBLE_percent())

8.806271551101968
1.8933333333333333
